In [ ]:
# Paste your text between the triple quotes

text = """
"""

In [ ]:
import re


# Regular expression to find IP addresses (IPv4 and IPv6)
ip_pattern = r'([0-9a-fA-F:.]+)'

# Find all IPs
ips = re.findall(ip_pattern, text)

# Clean and filter IPs
filtered_ips = []
for ip in ips:
    if '.' in ip :
        # strip non ip chars
        ip_cleaned = re.sub(r'[^0-9.]', '', ip)
        # if it's an IPv4 address, check if it has 4 octets
        octets = ip_cleaned.split('.')
        if len(octets) == 4 and all(0 <= int(octet) < 256 for octet in octets):
            filtered_ips.append(ip_cleaned)
    elif ':' in ip:
        # strip non ip chars
        ip_cleaned = re.sub(r'[^0-9a-fA-F:]', '', ip)
        # if it's an IPv6 address, check if it has valid format
        if re.match(r'^[0-9a-fA-F:]+$', ip_cleaned):
            # Check if it has valid number of groups (8 groups of 4 hex digits or less)
            groups = ip_cleaned.split(':')
            # get only first 8 groups
            if len(groups) > 8:
                groups = groups[:8]
            if len(groups) <= 8:
                # Check for empty groups (compressed format)
                if '' in groups:
                    if groups.count('') > 1:
                        continue
                # If there are empty groups, ensure the total number of groups is valid
                if len(groups) > 8:
                    continue
                # Check if each group has valid length (1 to 4 hex digits)
                if all(0 < len(group) <= 4 for group in groups if group != ''):
                    filtered_ips.append(':'.join(groups))
                    

# Remove duplicates
unique_ips = sorted(set(filtered_ips))

# Display the result
for ip in unique_ips:
    print(ip)


In [ ]:
import socket
import requests


def get_country_from_ip(ip_address):
    """
    Retrieves the country associated with an IP address using an external API.

    Args:
        ip_address (str): The IP address to look up.

    Returns:
        str or None: The two-letter country code (e.g., "US", "CZ") if found,
                     None otherwise or if an error occurs.
    """
    try:
        response = requests.get(f"http://ip-api.com/json/{ip_address}")
        response.raise_for_status()  # Raise an exception for bad status codes
        data = response.json()
        if data.get("status") == "success":
            return data.get("countryCode")
        else:
            print(f"Error looking up IP {ip_address}: {data.get('message')}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error during IP lookup for {ip_address}: {e}")
        return None
    
def check_dnsbl(ip_address, dnsbl_domain):
    """Checks if an IP address is listed on a DNSBL."""
    try:
        reversed_ip = ".".join(reversed(ip_address.split(".")))
        lookup_domain = f"{reversed_ip}.{dnsbl_domain}"
        result = socket.gethostbyname(lookup_domain)
        return True, result
    except socket.gaierror:
        return False, None

spamhaus_zen = "zen.spamhaus.org"

for ip_to_check in unique_ips:
    # Get the country code for the IP address
    country_code = get_country_from_ip(ip_to_check)
    # Check if the IP is listed on Spamhaus ZEN
    listed, result = check_dnsbl(ip_to_check, spamhaus_zen)

    #  ufw allow from 192.168.1.100 comment 'Allowed for trusted client'
    print(f"sudo ufw deny from {ip_to_check} comment 'Blocked {country_code}, {spamhaus_zen}: {result}'") 
      